In [69]:
# Librerías necesarias para la limpieza y transformación de datos
import pandas as pd

## Carga del dataset y creación de la copia de trabajo

In [70]:
# Cargamos el dataset original
df_original = pd.read_csv('../hr.csv')

# Creamos una copia de trabajo para no modificar nunca el archivo original
# Regla de oro: todas las transformaciones se hacen sobre df_clean
df_clean = df_original.copy()

# Confirmamos dimensiones
print(f"Filas: {df_clean.shape[0]}")
print(f"Columnas: {df_clean.shape[1]}")
df_clean.head()

Filas: 1474
Columnas: 35


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80.0,1,6,3.0,3,2,2,2,2.0


## Copiamos el DF para no modificar el original.

In [71]:
# Cargamos los datos originales subiendo un nivel de carpeta con '../'
df_original = pd.read_csv('../hr.csv')

# Creamos la copia de trabajo
df_clean = df_original.copy()

# Comprobamos que carga mostrando las primeras filas
df_clean.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41.0,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80.0,0,8,0.0,1,6,4,0,5.0
1,49.0,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,NaN,1,10,3.0,3,10,7,1,7.0
2,37.0,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,NaN,0,7,3.0,3,0,0,0,0.0
3,33.0,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80.0,0,8,3.0,3,8,7,3,0.0
4,27.0,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80.0,1,6,3.0,3,2,2,2,2.0


## Eliminamos columnas constantes y filas duplicadas

In [72]:
def drop_useless_cols(df):
    """
    Elimina las columnas constantes que no aportan
    variabilidad ni valor al análisis.
    Las detecta automáticamente en vez de usar una lista fija.
    """
    df_mod = df.copy()
    # Detectamos automáticamente las columnas con un solo valor único
    cols_constantes = [col for col in df_mod.columns if df_mod[col].nunique() == 1]
    return df_mod.drop(columns=cols_constantes, errors='ignore')

def drop_duplicates(df):
    """
    Elimina las filas duplicadas del dataset.
    """
    df_dropped = df.drop_duplicates()
    return df_dropped

# Aplicamos las dos funciones
df_clean = drop_useless_cols(df_clean)
df_clean = drop_duplicates(df_clean)

# Verificamos el resultado
print(f"Columnas originales: {df_original.shape[1]}")
print(f"Columnas tras limpieza: {df_clean.shape[1]}")
print(f"Filas originales: {df_original.shape[0]}")
print(f"Filas tras eliminar duplicados: {df_clean.shape[0]}")

Columnas originales: 35
Columnas tras limpieza: 32
Filas originales: 1474
Filas tras eliminar duplicados: 1470


## Normalizamos el formato de JobRole

In [73]:
def fix_job_role(df):
    """
    Elimina espacios en blanco en los extremos de 'JobRole' 
    y transforma el texto a formato título estándar.
    """
    df_mod = df.copy()
    df_mod['JobRole'] = df_mod['JobRole'].str.strip().str.title()
    return df_mod

# Aplicamos la función
df_clean = fix_job_role(df_clean)

# Verificamos el resultado
print("--- Valores de JobRole corregidos ---")
print(df_clean['JobRole'].unique())

--- Valores de JobRole corregidos ---
<StringArray>
[          'Sales Executive',        'Research Scientist',
     'Laboratory Technician',    'Manufacturing Director',
 'Healthcare Representative',                   'Manager',
      'Sales Representative',         'Research Director',
           'Human Resources']
Length: 9, dtype: str


## Corregimos la errata en MaritalStatus

In [74]:
def fix_marital_status(df):
    """
    Corrige la errata de escritura 'Marreid' pasándola a 'Married'
    en la columna MaritalStatus.
    """
    df_mod = df.copy()
    correcciones = {'Marreid': 'Married'}
    df_mod['MaritalStatus'] = df_mod['MaritalStatus'].replace(correcciones)
    return df_mod

# Aplicamos la función
df_clean = fix_marital_status(df_clean)

# Verificamos el resultado
# Nota: el nan que aparece aquí se tratará en la siguiente celda
print("--- Valores de MaritalStatus corregidos ---")
print(df_clean['MaritalStatus'].unique())

--- Valores de MaritalStatus corregidos ---
<StringArray>
['Single', 'Married', 'Divorced', nan]
Length: 4, dtype: str


## Tratamos los valores nulos

In [75]:
def fix_categorical_nulls(df):
    """
    Rellena los valores nulos de las columnas categóricas con 'Unknown'.
    """
    df_mod = df.copy()
    cols_with_nulls = ['Department', 'MaritalStatus', 'OverTime', 'BusinessTravel']
    for col in cols_with_nulls:
        df_mod[col] = df_mod[col].fillna('Unknown')
    return df_mod

def fix_numerical_nulls(df):
    """
    Rellena los valores nulos de las columnas numéricas con la mediana.
    Usamos la mediana porque es resistente a valores extremos (outliers).
    """
    df_mod = df.copy()
    cols_with_nulls = ['Age', 'JobSatisfaction', 'MonthlyIncome', 
                       'TrainingTimesLastYear', 'YearsWithCurrManager']
    for col in cols_with_nulls:
        median_val = df_mod[col].median()
        df_mod[col] = df_mod[col].fillna(median_val)
    return df_mod

# Aplicamos las dos funciones
df_clean = fix_categorical_nulls(df_clean)
df_clean = fix_numerical_nulls(df_clean)

# Verificamos que no queda ningún nulo en las columnas tratadas
print("--- Nulos restantes en columnas categóricas ---")
print(df_clean[['Department', 'MaritalStatus', 'OverTime', 'BusinessTravel']].isnull().sum())
print("\n--- Nulos restantes en columnas numéricas ---")
print(df_clean[['Age', 'JobSatisfaction', 'MonthlyIncome', 
                'TrainingTimesLastYear', 'YearsWithCurrManager']].isnull().sum())

--- Nulos restantes en columnas categóricas ---
Department        0
MaritalStatus     0
OverTime          0
BusinessTravel    0
dtype: int64

--- Nulos restantes en columnas numéricas ---
Age                      0
JobSatisfaction          0
MonthlyIncome            0
TrainingTimesLastYear    0
YearsWithCurrManager     0
dtype: int64


## Estandarizamos columnas binarias (Codificación a 1 / 0)

In [76]:
def encode_binary(df):
    """
    Codifica las columnas binarias 'Attrition' y 'OverTime' a 1/0.
    Yes → 1, No → 0, Unknown → nulo (pd.NA).
    """
    df_mod = df.copy()
    
    binary_mapping = {'Yes': 1, 'No': 0, 'Unknown': pd.NA}
    
    df_mod['Attrition'] = df_mod['Attrition'].replace(binary_mapping).astype('Int64')
    df_mod['OverTime'] = df_mod['OverTime'].replace(binary_mapping).astype('Int64')
    
    return df_mod

# Aplicamos la función
df_clean = encode_binary(df_clean)

# Verificamos el resultado
print("--- Valores únicos finales de Attrition ---")
print(df_clean['Attrition'].unique())
print("\n--- Valores únicos finales de OverTime ---")
print(df_clean['OverTime'].unique())

--- Valores únicos finales de Attrition ---
<IntegerArray>
[1, 0]
Length: 2, dtype: Int64

--- Valores únicos finales de OverTime ---
<IntegerArray>
[1, 0, <NA>]
Length: 3, dtype: Int64


## Corregimos los tipos de datos

In [77]:
def fix_dtypes(df):
    """
    Convierte columnas numéricas que están en float64
    a int64, ya que representan valores enteros.
    Solo se aplica después de haber tratado los nulos.
    """
    df_mod = df.copy()
    cols_to_fix = ['Age', 'JobSatisfaction', 'MonthlyIncome', 
                   'TrainingTimesLastYear', 'YearsWithCurrManager']
    for col in cols_to_fix:
        df_mod[col] = df_mod[col].astype(int)
    return df_mod

# Aplicamos la función
df_clean = fix_dtypes(df_clean)

# Verificamos que los tipos son correctos ahora
print(df_clean[['Age', 'JobSatisfaction', 'MonthlyIncome', 
                'TrainingTimesLastYear', 'YearsWithCurrManager']].dtypes)

Age                      int64
JobSatisfaction          int64
MonthlyIncome            int64
TrainingTimesLastYear    int64
YearsWithCurrManager     int64
dtype: object


## Pipeline completo

In [79]:
def limpiar_dataset(df):
    """
    Encadena todas las transformaciones en orden.
    La salida es el dataset limpio y listo para la Fase 3.
    """
    df_clean = df.copy()
    df_clean = drop_useless_cols(df_clean)
    df_clean = drop_duplicates(df_clean)
    df_clean = fix_job_role(df_clean)
    df_clean = fix_marital_status(df_clean)
    df_clean = fix_categorical_nulls(df_clean)
    df_clean = fix_numerical_nulls(df_clean)
    df_clean = fix_dtypes(df_clean)
    df_clean = encode_binary(df_clean)
    return df_clean

# Ejecutamos el pipeline completo
df_clean = limpiar_dataset(df_original)

## Control de Calidad

In [80]:
# --- Control de calidad automático ---

# 1. Dimensiones del dataset
assert df_clean.shape[1] == 32, f"Error: Se esperaban 32 columnas pero hay {df_clean.shape[1]}"
assert df_clean.shape[0] == 1470, f"Error: Se esperaban 1470 filas pero hay {df_clean.shape[0]}"

# 2. Nulos en columnas categóricas
assert df_clean['Department'].isnull().sum() == 0, "Error: Siguen quedando nulos en Department"
assert df_clean['MaritalStatus'].isnull().sum() == 0, "Error: Siguen quedando nulos en MaritalStatus"
assert df_clean['BusinessTravel'].isnull().sum() == 0, "Error: Siguen quedando nulos en BusinessTravel"

# 3. Nulos en columnas numéricas
assert df_clean['Age'].isnull().sum() == 0, "Error: Siguen quedando nulos en Age"
assert df_clean['MonthlyIncome'].isnull().sum() == 0, "Error: Siguen quedando nulos en MonthlyIncome"
assert df_clean['TrainingTimesLastYear'].isnull().sum() == 0, "Error: Siguen quedando nulos en TrainingTimesLastYear"
assert df_clean['YearsWithCurrManager'].isnull().sum() == 0, "Error: Siguen quedando nulos en YearsWithCurrManager"

# 4. Errata corregida
assert 'Marreid' not in df_clean['MaritalStatus'].unique(), "Error: La errata 'Marreid' sigue existiendo"

# 5. Tipos de datos correctos
assert df_clean['Age'].dtype == 'int64', "Error: Age no es int64"
assert df_clean['MonthlyIncome'].dtype == 'int64', "Error: MonthlyIncome no es int64"
assert df_clean['Attrition'].dtype == 'Int64', "Error: Attrition no es Int64"
assert df_clean['OverTime'].dtype == 'Int64', "Error: OverTime no es Int64"

print("✅ ¡Todos los controles de calidad han pasado! El dataset está limpio y listo.")

✅ ¡Todos los controles de calidad han pasado! El dataset está limpio y listo.


## Guardamos el dataset limpio

In [ ]:
# Guardamos el dataset limpio en un nuevo CSV sin el índice
df_clean.to_csv('Fase2_ParejaB_limpiezaJezi.csv', index=False)

print("✅ Archivo 'Fase2_ParejaB_limpiezaJezi.csv' guardado con éxito.")
print(f"Dimensiones finales del dataset limpio: {df_clean.shape[0]} filas x {df_clean.shape[1]} columnas")

# Vista previa del resultado final
df_clean.head()

✅ Archivo 'Fase2_ParejaB_limpiezaJezi.csv' guardado con éxito.
Dimensiones finales del dataset limpio: 1470 filas x 32 columnas


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,3,4,1,6,3,3,2,2,2,2
